# Emission Line Maps From Galaxy Particle Distributions

Synthesizer can create resolved emission line maps, in addition to the
photometric images demonstrated in the
[particle imaging notebook](particle_imaging.ipynb).

Line mapping mirrors photometric imaging almost exactly: rather than
projecting filter-weighted luminosities/fluxes into an ``Image`` per filter,
we project per-line luminosities/fluxes into an ``Image`` (a line map) per
requested line id, using a ``LineMapper`` in place of a
``PhotometricImager``. Everything else described in the particle imaging
notebook (PSF application, noise, RGB images, angular vs Cartesian imaging,
...) works identically for line maps, just keyed by line id instead of
filter code.

Below we demonstrate the core workflow using the same CAMELS galaxy as the
particle imaging notebook.

In [ ]:
import matplotlib.colors as cm
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
from astropy.cosmology import Planck18 as cosmo
from unyt import angstrom, kpc

from synthesizer import TEST_DATA_DIR
from synthesizer.emission_models import BimodalPacmanEmission
from synthesizer.emission_models.attenuation import PowerLaw
from synthesizer.grid import Grid
from synthesizer.instruments import LineMapper
from synthesizer.kernel_functions import Kernel
from synthesizer.load_data.load_camels import load_CAMELS_IllustrisTNG

# Define the grid
grid_name = "test_grid"
grid = Grid(grid_name, new_lam=np.logspace(2, 5, 600) * angstrom)

# Create galaxy object
gal = load_CAMELS_IllustrisTNG(
    TEST_DATA_DIR,
    snap_name="camels_snap.hdf5",
    group_name="camels_subhalo.hdf5",
    physical=True,
)[1]

### Getting the lines

To make a line map we need per-particle emission line luminosities for
each star particle. To do this we use the galaxy's built in ``get_lines``
method and pass a ``per_particle`` model, exactly as ``get_spectra`` is used
for photometric imaging. This will generate a ``LineCollection`` for each
particle in the galaxy.

We use H-alpha and H-beta here: two of the most commonly used emission lines
in the literature (star formation rate indicators, the Balmer decrement for
dust attenuation, BPT diagrams, etc.), and bright enough in this galaxy to
show clear structure.

In [ ]:
# Get the stellar pacman model - the same model used for photometric imaging,
# just queried for lines instead of (or as well as) spectra
model = BimodalPacmanEmission(
    grid=grid,
    tau_v_ism=1.0,
    tau_v_birth=0.7,
    dust_curve_ism=PowerLaw(slope=-1.3),
    dust_curve_birth=PowerLaw(slope=-0.7),
    fesc=0.1,
    fesc_ly_alpha=0.9,
    label="total",
    per_particle=True,
)

# H-alpha and H-beta
line_ids = ["H 1 6562.80A", "H 1 4861.32A"]

# Generate the per-particle lines
lines = gal.get_lines(line_ids, model)
print(lines)

## Mapping

The last step before we can make any line maps is to define a
``LineMapper`` with the resolution and line ids attached, plus the FOV, or
width, of the maps. A ``LineMapper`` is configured with ``line_ids``
instead of ``filters`` — everything else about the instrument works
identically to a ``PhotometricImager``.

In [ ]:
# Define the width and resolution of the map
width = 30 * kpc
resolution = width / 200

# Create a line mapping instrument
instrument = LineMapper(
    label="DemoLineMapper", resolution=resolution, line_ids=line_ids
)

print(f"Map width is {width:.2f} with {resolution:.2f} resolution")

Now we have everything we need to make line maps. The main public
interface is the high-level galaxy API: ``get_line_maps_luminosity`` for
luminosity maps and ``get_line_maps_flux`` for flux maps, exactly
mirroring ``get_images_luminosity``/``get_images_flux`` for photometric
images. Both take the map properties defined above, the ``line_ids`` we
want maps for, and the emission model label the lines were generated with
("total" in this case), and return a single ``ImageCollection`` containing
one line map (``Image``) per requested line id.

In [ ]:
# Get the SPH kernel
kernel_data = Kernel().get_kernel()

# Get the histogram line maps
hist_maps = gal.get_line_maps_luminosity(
    "total",
    line_ids=line_ids,
    instrument=instrument,
    fov=width,
    img_type="hist",
    kernel=kernel_data,
    kernel_threshold=1,
    cosmo=cosmo,
)

# Get the smoothed line maps
smooth_maps = gal.get_line_maps_luminosity(
    "total",
    line_ids=line_ids,
    instrument=instrument,
    fov=width,
    img_type="smoothed",
    kernel=kernel_data,
    kernel_threshold=1,
    cosmo=cosmo,
)

Maps generated using these getter methods are both returned and attached
to the `Galaxy` and components depending on where the lines were calculated.
In the example above, the "total" lines were calculated on the `Stars`
component, so the maps are attached to `gal.stars` under
`gal.stars.line_maps_lnu`.

In [ ]:
# Lets set up a simple normalisation across all maps
vmax = 0
for img in hist_maps.values():
    up = np.percentile(img.arr, 99.9)
    if up > vmax:
        vmax = up
hist_norm = cm.Normalize(vmin=0, vmax=vmax)
vmax = 0
for img in smooth_maps.values():
    up = np.percentile(img.arr, 99.9)
    if up > vmax:
        vmax = up
smooth_norm = cm.Normalize(vmin=0, vmax=vmax)


# Set up plot
fig = plt.figure(figsize=(4 * len(line_ids), 4 * 2))
gs = gridspec.GridSpec(2, len(line_ids), hspace=0.0, wspace=0.0)

# Create top row
axes = []
for i in range(len(line_ids)):
    axes.append(fig.add_subplot(gs[0, i]))

# Loop over maps plotting them
for ax, line_id in zip(axes, line_ids):
    ax.imshow(hist_maps[line_id].arr, norm=hist_norm, cmap="Greys_r")
    ax.set_title(line_id)
    ax.tick_params(
        axis="both",
        which="both",
        left=False,
        right=False,
        labelleft=False,
        labelright=False,
        bottom=False,
        top=False,
        labelbottom=False,
        labeltop=False,
    )

# Set y axis label on left most plot
axes[0].set_ylabel("Histogram")

# Create bottom row
axes = []
for i in range(len(line_ids)):
    axes.append(fig.add_subplot(gs[1, i]))

# Loop over maps plotting them
for ax, line_id in zip(axes, line_ids):
    ax.imshow(smooth_maps[line_id].arr, norm=smooth_norm, cmap="Greys_r")
    ax.tick_params(
        axis="both",
        which="both",
        left=False,
        right=False,
        labelleft=False,
        labelright=False,
        bottom=False,
        top=False,
        labelbottom=False,
        labeltop=False,
    )

# Set y axis label on left most plot
axes[0].set_ylabel("Smoothed")

# Plot the map
plt.show()
plt.close(fig)

## Flux maps

Just as with photometric images, we can also make flux line maps using
``get_line_maps_flux``, once the observed lines have been calculated with
``get_observed_lines``.

In [ ]:
gal.get_observed_lines(cosmo)

flux_maps = gal.get_line_maps_flux(
    "total",
    line_ids=line_ids,
    instrument=instrument,
    fov=width,
    img_type="smoothed",
    kernel=kernel_data,
    kernel_threshold=1,
    cosmo=cosmo,
)

fig, ax = flux_maps.plot_images(show=True)
plt.close(fig)

That's the core line mapping workflow. For PSF application, noise, RGB
maps, and angular vs Cartesian imaging — all of which work identically for
line maps, just keyed by line id instead of filter code — see the
[particle imaging notebook](particle_imaging.ipynb).